In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from fpdf import FPDF
from datetime import datetime, timedelta
import os

# ==========================================
# 1. MAESTROS DE CONFIGURACIÓN (Lógica del SQL)
# ==========================================
MAESTRO_FORMULAS = {
    '2110': [50, 150, 10, 40], '2117': [50, 150, 10, 40], '2125': [50, 150, 10, 40],
    '2126': [50, 150, 10, 40], '2127': [50, 150, 10, 40], '2128': [50, 150, 10, 40],
    '2129': [50, 150, 10, 40], '2130': [50, 150, 10, 40], '2140': [50, 150, 10, 40],
    '2141': [50, 150, 10, 40], '2142': [50, 150, 10, 40], '2146': [50, 150, 10, 40],
    '2147': [50, 150, 10, 40], '2148': [50, 150, 10, 40], '2149': [50, 150, 10, 40],
    '2157': [50, 150, 10, 40], '2254': [65, 150, 10, 40], '2260': [65, 150, 10, 40],
    '2263': [65, 150, 10, 40], '2267': [65, 150, 10, 40], '3240': [60, 150, 3, 20],
    '3360': [60, 150, 3, 20],  '2206': [65, 150, 10, 40], '2261': [65, 150, 10, 40],
    '3346': [50, 150, 10, 40], '3352': [50, 150, 10, 40], '3366': [55, 150, 10, 40],
    '3367': [55, 150, 10, 40], '3368': [55, 150, 10, 40], '3370': [55, 150, 10, 40]
}

MAESTRO_EQUIPOS = {'310': 200, '320': 200, '330': 100, '340': 100, '350': 100}

def load_and_process_data():
    df1 = pd.read_csv('Planta1.csv', low_memory=False)
    df2 = pd.read_csv('Planta2.csv', low_memory=False)
    df = pd.concat([df1, df2], ignore_index=True)
    
    df['fecha_registro'] = pd.to_datetime(df['fecha_registro'])
    df['id_formula'] = df['des_codigo_producto'].astype(str).str[:4]
    
    df.loc[df['promedio_temp_acondicionador'] < 0, 'promedio_temp_acondicionador'] = np.nan
    df.loc[df['presion_vapor'] < 0, 'presion_vapor'] = np.nan

    df['umbral_carga'] = df['des_peletizadora'].map(MAESTRO_EQUIPOS).fillna(80)
    
    df = df.sort_values(by=['des_peletizadora', 'numero_orden_peletizado', 'fecha_registro'])
    
    df['corriente_suave'] = df.groupby(['des_peletizadora', 'numero_orden_peletizado'])['corriente'].transform(lambda x: x.ewm(alpha=0.33, adjust=False).mean())
    df['temp_suave'] = df.groupby(['des_peletizadora', 'numero_orden_peletizado'])['promedio_temp_acondicionador'].transform(lambda x: x.ewm(alpha=0.33, adjust=False).mean())
    df['presion_suave'] = df.groupby(['des_peletizadora', 'numero_orden_peletizado'])['presion_vapor'].transform(lambda x: x.ewm(alpha=0.66, adjust=False).mean())

    df_operativo = df[df['corriente_suave'] >= df['umbral_carga']].copy()
    
    return df, df_operativo

# ==========================================
# 2. MOTOR DE SIMULACIÓN DE ALARMAS Y FATIGA
# ==========================================
def calculate_alarm_fatigue(df_operativo):
    results = []
    
    for maquina, group in df_operativo.groupby('des_peletizadora'):
        state = 'OK'
        alarm_times = []
        
        for idx, row in group.iterrows():
            formula = row['id_formula']
            if formula not in MAESTRO_FORMULAS:
                continue
            
            tmin, tmax, pmin, pmax = MAESTRO_FORMULAS[formula]
            temp = row['temp_suave']
            pres = row['presion_suave']
            
            is_oob = False
            if pd.notnull(temp) and (temp < tmin or temp > tmax): is_oob = True
            if pd.notnull(pres) and (pres < pmin or pres > pmax): is_oob = True
                
            row_out = 1 if is_oob else 0
            triggered_alarm = 0
            current_time = row['fecha_registro']
            
            if is_oob and state == 'OK':
                alarm_times = [t for t in alarm_times if current_time - t <= timedelta(minutes=10)]
                if len(alarm_times) < 2:
                    triggered_alarm = 1
                    alarm_times.append(current_time)
                    state = 'ALARM'
            elif not is_oob and state == 'ALARM':
                state = 'OK'
                
            results.append({'id_formula': formula, 'desvio': row_out, 'alarma': triggered_alarm})

    df_res = pd.DataFrame(results)
    if df_res.empty: return pd.DataFrame()
    
    # Sumarizamos desvíos y alarmas
    summary = df_res.groupby('id_formula').sum().reset_index()
    summary['ruido_evitado'] = np.where(
        summary['desvio'] > 0, 
        ((summary['desvio'] - summary['alarma']) / summary['desvio'] * 100).round(1), 
        0
    )
    
    # --- NUEVO: CALCULAR ESTADÍSTICAS REALES (MEDIA Y STD) ---
    stats = df_operativo.groupby('id_formula').agg(
        temp_mean=('temp_suave', 'mean'),
        temp_std=('temp_suave', 'std'),
        pres_mean=('presion_suave', 'mean'),
        pres_std=('presion_suave', 'std')
    ).reset_index()
    
    # Cruzamos las estadísticas con el resumen de alarmas
    summary = pd.merge(summary, stats, on='id_formula', how='left')
    
    return summary[summary['desvio'] > 0].sort_values('desvio', ascending=False)

# ==========================================
# 3. GENERADORES DE GRÁFICOS
# ==========================================
def create_real_pareto_chart(df):
    fig, ax = plt.subplots(figsize=(7, 4))
    alertas_temp, alertas_presion = 0, 0
    
    for formula, limits in MAESTRO_FORMULAS.items():
        data_f = df[df['id_formula'] == formula]
        if not data_f.empty:
            tmin, tmax, pmin, pmax = limits
            alertas_temp += ((data_f['temp_suave'] < tmin) | (data_f['temp_suave'] > tmax)).sum()
            alertas_presion += ((data_f['presion_suave'] < pmin) | (data_f['presion_suave'] > pmax)).sum()
            
    alertas_corriente = (df['corriente'] < df['umbral_carga']).sum() 

    alerts = {'Desvio Temperatura (Real)': alertas_temp, 'Desvio Presion (Real)': alertas_presion, 'Caidas de Corriente': alertas_corriente}
    alert_df = pd.Series(alerts).sort_values(ascending=False)
    sns.barplot(x=alert_df.index, y=alert_df.values, ax=ax, palette='Reds_r')
    ax.set_title('Top Desviaciones (Basado en Limites Maestro)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Cantidad de Eventos')
    
    file_name = 'temp_pareto.png'
    plt.tight_layout()
    plt.savefig(file_name, format='png', dpi=150)
    plt.close()
    return file_name

def create_simulated_scatter(df):
    fig, ax = plt.subplots(figsize=(7, 4))
    df_quality = pd.DataFrame({
        'tiempo_proceso': np.random.uniform(15, 45, 100), 
        'porcentaje_durabilidad': np.random.uniform(88, 98, 100)
    })
        
    sns.scatterplot(data=df_quality, x='tiempo_proceso', y='porcentaje_durabilidad', alpha=0.6, ax=ax, color='purple')
    ax.set_title('Impacto de Retencion en Durabilidad * (DATOS SIMULADOS)', fontsize=12, fontweight='bold', color='purple')
    ax.set_xlabel('Tiempo de Proceso (Segundos)')
    ax.set_ylabel('Durabilidad (%)')
    
    file_name = 'temp_scatter.png'
    plt.tight_layout()
    plt.savefig(file_name, format='png', dpi=150)
    plt.close()
    return file_name

def create_discovery_boxplot(df, formula_target):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    data = df[df['id_formula'] == formula_target]
    
    sns.boxplot(y=data['temp_suave'], ax=ax1, color='orange')
    ax1.set_title('Comportamiento Termico Real (C)')
    ax1.set_ylabel('Temperatura Suavizada')
    
    sns.boxplot(y=data['presion_suave'], ax=ax2, color='lightblue')
    ax2.set_title('Comportamiento de Presion (PSI)')
    ax2.set_ylabel('Presion Suavizada')
    
    file_name = 'temp_boxplot.png'
    plt.tight_layout()
    plt.savefig(file_name, format='png', dpi=150)
    plt.close()
    return file_name

# ==========================================
# 4. MOTOR DEL PDF (FPDF)
# ==========================================
class PDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 14)
        self.set_text_color(40, 40, 40)
        self.cell(0, 10, 'REPORTE ANALITICO DE PRODUCCION (OEE & IA)', 0, 1, 'C')
        self.set_font('Arial', 'I', 9)
        self.set_text_color(100, 100, 100)
        self.cell(0, 5, f'Generado Automaticamente - {datetime.now().strftime("%Y-%m-%d %H:%M")}', 0, 1, 'C')
        self.ln(5)

    def chapter_title(self, title, is_simulated=False):
        self.set_font('Arial', 'B', 12)
        if is_simulated:
            self.set_fill_color(240, 230, 255)
            self.set_text_color(100, 0, 150)
        else:
            self.set_fill_color(200, 220, 255)
            self.set_text_color(0, 0, 0)
            
        self.cell(0, 8, title, 0, 1, 'L', fill=True)
        self.set_text_color(0, 0, 0)
        self.ln(4)
        
    def stat_box(self, label, value, is_simulated=False):
        self.set_font('Arial', 'B', 10)
        if is_simulated:
            label = f"{label} *"
        self.cell(70, 8, label + ":", border=1)
        self.set_font('Arial', '', 10)
        self.cell(40, 8, str(value), border=1, ln=1)

def generate_pdf(df_crudo, df_operativo, df_fatiga):
    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # -----------------------------------------------------
    # PÁGINA 1: SNAPSHOT
    # -----------------------------------------------------
    pdf.add_page()
    pdf.chapter_title('1. Resumen Ejecutivo (Datos Reales de Planta)')
    
    df_core = df_operativo[df_operativo['id_formula'].isin(MAESTRO_FORMULAS.keys())]
    total_orders = df_core['numero_orden_peletizado'].nunique()
    estabilidad_pct = round((df_core[df_core['retornando'] == 0].shape[0] / max(df_core.shape[0], 1)) * 100, 1)
    
    pdf.set_font('Arial', '', 11)
    pdf.multi_cell(0, 6, 'Resumen basado estrictamente en la telemetria real de las maquinas y las 24 formulas objetivo, filtrando momentos de maquinas apagadas.')
    pdf.ln(5)
    
    if not df_core.empty:
        pdf.stat_box('Ventana de Tiempo', f"{df_core['fecha_registro'].min().strftime('%Y-%m-%d')} a {df_core['fecha_registro'].max().strftime('%Y-%m-%d')}")
    pdf.stat_box('Ordenes Procesadas (Top 24)', total_orders)
    pdf.stat_box('Tasa de Estabilidad Real', f"{estabilidad_pct} %")
    
    maquina_critica = str(df_core['des_peletizadora'].value_counts().index[0]) if not df_core.empty else "N/A"
    pdf.stat_box('Maquina con Mas Procesos', maquina_critica)

    # -----------------------------------------------------
    # PÁGINA 2: FEEDBACK LOOP
    # -----------------------------------------------------
    pdf.add_page()
    pdf.chapter_title('2. Desempeno del Agente IA y Feedback Loop *', is_simulated=True)
    pdf.set_font('Arial', 'I', 10)
    pdf.multi_cell(0, 6, '(*) ADVERTENCIA: Los datos en esta seccion son SUPUESTOS SIMULADOS para propositos de diseno. En produccion, provendran del archivo "historial_alertas.csv".')
    pdf.ln(5)
    
    pdf.set_font('Arial', '', 11)
    pdf.stat_box('Tasa de Respuesta Operador', '84.5 %', is_simulated=True)
    pdf.stat_box('Aciertos LLM (Util y Aplicado)', '62.0 %', is_simulated=True)
    pdf.stat_box('Falsos Positivos (Ruido)', '23.0 %', is_simulated=True)
    pdf.stat_box('Fallas Fisicas/Mecanicas', '15.0 %', is_simulated=True)
    
    pdf.ln(5)
    pdf.set_font('Arial', 'B', 11)
    pdf.cell(0, 8, 'Precision del Agente (Accuracy Rate)*: 73.3%', 0, 1)

    # -----------------------------------------------------
    # PÁGINA 3: ANALÍTICA DE PROCESO
    # -----------------------------------------------------
    pdf.add_page()
    pdf.chapter_title('3. Analitica de Proceso y Desviaciones (Real)')
    pdf.set_font('Arial', '', 11)
    pdf.multi_cell(0, 6, 'Este Pareto calcula los desvios operativos reales evaluando la telemetria suavizada de las maquinas contra los limites estaticos del Maestro de Formulas.')
    
    file_pareto = create_real_pareto_chart(df_core)
    pdf.image(file_pareto, x=15, w=150)

    # -----------------------------------------------------
    # PÁGINA 4: FATIGA DE ALARMAS (NUEVA TABLA CORREGIDA)
    # -----------------------------------------------------
    pdf.add_page()
    pdf.chapter_title('4. Analisis de Fatiga de Alarmas y Eficiencia del Agente IA')
    
    pdf.set_font('Arial', '', 10)
    pdf.multi_cell(0, 6, 'Regla Aplicada: Maximo 2 notificaciones en una ventana de 10 minutos. Se expone la desviacion estadistica (Media +- Std) vs el Rango de la Receta.')
    pdf.ln(3)
    
    # Renderizamos la Tabla (Ajustada para que quepan 8 columnas en una página normal)
    pdf.set_fill_color(50, 50, 50)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font('Arial', 'B', 7) # Letra más pequeña para que quepa bien
    
    # Anchos milimétricos (Total debe ser ~180 para una hoja A4 normal)
    col_widths = [15, 18, 28, 18, 28, 20, 20, 33]
    headers = ['Formula', 'Rango T', 'Temp (Real)', 'Rango P', 'Pres (Real)', 'Desvios', 'Alarmas', 'Ahorro Ruido (%)']
    
    for i, header in enumerate(headers):
        pdf.cell(col_widths[i], 8, header, 1, 0, 'C', fill=True)
    pdf.ln()
    
    pdf.set_text_color(0, 0, 0)
    pdf.set_font('Arial', '', 8)
    
    # Llenamos datos de la tabla (Top 15 para cuidar el tamaño de página)
    for idx, row in df_fatiga.head(20).iterrows():
        f_id = str(row['id_formula'])
        
        # Recuperamos los rangos ideales del Maestro
        tmin, tmax, pmin, pmax = MAESTRO_FORMULAS.get(f_id, [0,0,0,0])
        rango_t = f"[{tmin}-{tmax}]"
        rango_p = f"[{pmin}-{pmax}]"
        
        # Formateamos Media +- Std
        t_m = row['temp_mean']
        t_s = row['temp_std'] if pd.notna(row['temp_std']) else 0
        temp_str = f"{t_m:.1f} (+-{t_s:.1f})"
        
        p_m = row['pres_mean']
        p_s = row['pres_std'] if pd.notna(row['pres_std']) else 0
        pres_str = f"{p_m:.1f} (+-{p_s:.1f})"
        
        # Dibujamos las Celdas
        pdf.cell(col_widths[0], 8, f_id, 1, 0, 'C')            # Formula
        pdf.cell(col_widths[1], 8, rango_t, 1, 0, 'C')         # Rango Ideal Temp
        pdf.cell(col_widths[2], 8, temp_str, 1, 0, 'C')        # Temp Real
        pdf.cell(col_widths[3], 8, rango_p, 1, 0, 'C')         # Rango Ideal Pres
        pdf.cell(col_widths[4], 8, pres_str, 1, 0, 'C')        # Pres Real
        pdf.cell(col_widths[5], 8, str(int(row['desvio'])), 1, 0, 'C') # Desvíos Físicos
        
        pdf.set_font('Arial', 'B', 8)
        pdf.cell(col_widths[6], 8, str(int(row['alarma'])), 1, 0, 'C') # Alarmas Efectivas
        pdf.set_font('Arial', '', 8)
        
        # Verde si hay ahorro alto, texto normal si no
        pdf.set_text_color(0, 100, 0) if row['ruido_evitado'] > 50 else pdf.set_text_color(0, 0, 0)
        pdf.cell(col_widths[7], 8, f"{row['ruido_evitado']} %", 1, 0, 'C') # Ahorro Ruido
        pdf.set_text_color(0, 0, 0)
        pdf.ln()

    # -----------------------------------------------------
    # PÁGINA 5: CALIDAD PREDICTIVA
    # -----------------------------------------------------
    pdf.add_page()
    pdf.chapter_title('5. Calidad Predictiva y Eficiencia Energetica')
    
    kw_total = df_core['kw_h_proceso'].sum() / 60
    pdf.set_font('Arial', 'B', 11)
    pdf.cell(0, 8, f'Indice de Consumo Energetico Estimado (Real): {round(kw_total, 2)} kW/h', 0, 1)
    pdf.ln(2)
    
    pdf.set_font('Arial', 'I', 10)
    pdf.multi_cell(0, 6, 'El cruce con pruebas de laboratorio de Calidad es un supuesto simulado (*), a la espera de integrar etiquetas completas de humedad y durabilidad.')
    
    file_scatter = create_simulated_scatter(df_crudo)
    pdf.image(file_scatter, x=15, w=150)

    # -----------------------------------------------------
    # PÁGINA 6: DISCOVERY 
    # -----------------------------------------------------
    pdf.add_page()
    
    formulas_presentes = df_crudo['id_formula'].dropna().unique()
    formulas_huerfanas = [f for f in formulas_presentes if f not in MAESTRO_FORMULAS.keys() and f != 'nan']
    
    if formulas_huerfanas:
        target = formulas_huerfanas[0] 
        data_target = df_crudo[df_crudo['id_formula'] == target]
        
        pdf.set_fill_color(255, 200, 200) 
        pdf.chapter_title(f'6. OPORTUNIDAD: Formula No Configurada ({target})')
        pdf.set_font('Arial', '', 11)
        pdf.multi_cell(0, 6, f'Gobernanza (Dato Real): La formula {target} se proceso {len(data_target)} veces, pero NO existe en la lista oficial de 24 formulas maestras. '
                             'Aqui tienes la huella estadistica de como se comporto realmente en maquina:')
        pdf.ln(5)
        
        pdf.stat_box('Temp Media Real', f"{round(data_target['temp_suave'].mean(), 1)} C")
        pdf.stat_box('Presion Media Real', f"{round(data_target['presion_suave'].mean(), 1)} PSI")
        
        file_box = create_discovery_boxplot(df_crudo, target)
        pdf.image(file_box, x=15, w=170)
    else:
         pdf.chapter_title('6. OPORTUNIDAD: Gobernanza de Formulas')
         pdf.cell(0, 8, 'Todas las formulas procesadas cuentan con configuracion. !Excelente!', 0, 1)

    # Guardado Final y Limpieza
    pdf.output("Reporte_Eficiencia_Final.pdf")
    print("¡Reporte generado exitosamente como 'Reporte_Eficiencia_Final.pdf'!")
    
    for tmp_file in ['temp_pareto.png', 'temp_scatter.png', 'temp_boxplot.png']:
        if os.path.exists(tmp_file):
            os.remove(tmp_file)

# ==========================================
# 5. EJECUCIÓN PRINCIPAL
# ==========================================
if __name__ == '__main__':
    print("1. Cargando y cruzando telemetria...")
    df_crudo, df_operativo = load_and_process_data()
    
    print("2. Ejecutando simulacion de Fatiga de Alarmas (Ventana 10 mins)...")
    df_fatiga = calculate_alarm_fatigue(df_operativo)
    
    print("3. Generando Reporte PDF Completo...")
    generate_pdf(df_crudo, df_operativo, df_fatiga)

1. Cargando y cruzando telemetria...
2. Ejecutando simulacion de Fatiga de Alarmas (Ventana 10 mins)...
3. Generando Reporte PDF Completo...


C:\Users\BRAYAN\AppData\Local\Temp\ipykernel_18512\2791332892.py:181: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  self.set_font('Arial', 'B', 14)
C:\Users\BRAYAN\AppData\Local\Temp\ipykernel_18512\2791332892.py:183: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 10, 'REPORTE ANALITICO DE PRODUCCION (OEE & IA)', 0, 1, 'C')
C:\Users\BRAYAN\AppData\Local\Temp\ipykernel_18512\2791332892.py:184: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  self.set_font('Arial', 'I', 9)
C:\Users\BRAYAN\AppData\Local\Temp\ipykernel_18512\2791332892.py:186: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  self.cell(0, 5, f'Generado Automaticamente - {datetime.now().strftime("%Y

¡Reporte generado exitosamente como 'Reporte_Eficiencia_Final.pdf'!
